# Language network — verb-gen DLD/HSL/TD analyses

Three arms for the **language network** (frontal-putamen FC + language network size), assembled from the validated `stats/` scripts:

1. between-group **connectivity comparison** (Welch ANOVA),
2. **group x FC** predicting SDQ-emotional (NB, 95% CI),
3. **group x network size** predicting SDQ-emotional (NB, 95% CI).

Each code cell is self-contained (re-imports, reloads data) and also writes its PNG/CSV outputs to `results/` exactly as the scripts do. Run top-to-bottom, or any section on its own. Use the project venv as the kernel.

In [ ]:
%matplotlib inline
# figures render inline AND are still saved to results/ by each script

## 1. Between-group connectivity comparison (Welch ANOVA)

3-group omnibus (DLD/HSL/TD) on the single language frontal-putamen edge (L_44 pars opercularis <-> Language-14 medial/anterior L-putamen), Fisher-z, with Games-Howell / Dunn post-hoc. Single edge, so no multiple-comparison correction.

*Source: `stats/stats_group_language_connectivity.py` (adapted for inline display).*

In [ ]:
"""
stats_group_language_connectivity.py

## Author: Han Wang
### 2026-08-14: Initial version (language frontal-putamen arm).

Between-group comparison of the SINGLE language frontal-putamen FC edge

    Language-14_L-Ctx (L_44, pars opercularis) <-> Language-14_L-Putamen
    (medial/anterior left putamen)

across the three verb-gen groups (DLD, HSL, TD), mirroring the striatal arm's
stats_group_connectivity.py but for one edge (so no 9-tile FDR).

Design (n up to 144: DLD=53, HSL=27, TD=64)
-------------------------------------------
1. Gather every analysed subject's per-subject edge CSV
   (<sub>_language_putamen_FC.csv), merge group, Fisher-z (already stored as z).
2. OMNIBUS on Fisher-z:
     - Welch's ANOVA (unequal variance / unequal n) -> F, p, partial eta^2, omega^2.
     - Kruskal-Wallis as a rank-based robustness backup.
3. Protected POST-HOC (interpret only if omnibus p<.05 -- Fisher-protected):
     - Games-Howell (unequal-variance pairwise) with Hedges g: DLD-TD, HSL-TD, DLD-HSL.
     - Dunn (Holm) rank-based backup.
Test on Fisher-z, display r.

Input:  results/language_connectivity_outputs/sub-*/sub-*_language_putamen_FC.csv
        dat_verbgen_analysis_144.csv  (code -> group)
Output: results/language_connectivity_outputs/group_language_putamen_long.csv
        results/language_connectivity_outputs/group_language_putamen_omnibus.csv
        results/language_connectivity_outputs/group_language_putamen_posthoc.csv
        results/language_connectivity_outputs/group_language_putamen.png
"""

import glob
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from scipy import stats
import pingouin as pg
import scikit_posthocs as sp

PROJECT_DIR = "/home/hanwang/Apps/Programming/matlab-proj/PFM_MSHBM_MHVerbGen"
LANG_DIR = f"{PROJECT_DIR}/results/language_connectivity_outputs"
LISTCSV = ("/home/hanwang/Documents/Data/verb_gen_krishnan/"
           "behavioural_scq_sdq/dat_verbgen_analysis_144.csv")

GROUPS = ["DLD", "HSL", "TD"]                    # display order (clinical gradient)
GROUP_COLORS = {"DLD": "#d63031", "HSL": "#2ca02c", "TD": "#0984e3"}
CONTRASTS = [("DLD", "TD"), ("HSL", "TD"), ("DLD", "HSL")]
EDGE = "L-Putamen(Language) <-> L_44 (pars opercularis)"


def stars(p):
    return "***" if p < .001 else "**" if p < .01 else "*" if p < .05 else "n.s."


def omega_sq(groups):
    k = len(groups); n = sum(len(g) for g in groups)
    grand = np.concatenate(groups).mean()
    ss_b = sum(len(g) * (g.mean() - grand) ** 2 for g in groups)
    ss_w = sum(((g - g.mean()) ** 2).sum() for g in groups)
    ss_t = ss_b + ss_w
    ms_w = ss_w / (n - k)
    denom = ss_t + ms_w
    return (ss_b - (k - 1) * ms_w) / denom if denom > 0 else np.nan


# ============================================================
# 1. Gather per-subject edges -> long
# ============================================================
beh = pd.read_csv(LISTCSV)[["code", "group"]].copy()
beh["code"] = beh["code"].astype(str)
code2group = beh.set_index("code")["group"].to_dict()

rows, missing = [], []
for code, group in code2group.items():
    sub = f"sub-{code}"
    hits = glob.glob(f"{LANG_DIR}/{sub}/{sub}_language_putamen_FC.csv")
    if not hits:
        missing.append(sub); continue
    m = pd.read_csv(hits[0]).iloc[0]
    r = float(m["r"]); z = float(m["z"])
    rows.append(dict(subject=sub, code=code, group=group, r=r, z=z))

long = pd.DataFrame(rows)
if missing:
    print(f"WARNING: {len(missing)} subjects had no language edge CSV: {missing}")
counts = long["group"].value_counts().to_dict()
print(f"Loaded {len(long)} subjects: " + ", ".join(f"{g}={counts.get(g,0)}" for g in GROUPS))
long.to_csv(f"{LANG_DIR}/group_language_putamen_long.csv", index=False)
print(f"Saved: {LANG_DIR}/group_language_putamen_long.csv")

# ============================================================
# 2. Omnibus (Welch ANOVA + Kruskal) on Fisher-z
# ============================================================
arrs = [long[long.group == g]["z"].to_numpy() for g in GROUPS]
wa = pg.welch_anova(data=long, dv="z", between="group").iloc[0]
F, p_w, np2 = float(wa["F"]), float(wa["p_unc"]), float(wa["np2"])
ddof1, ddof2 = float(wa["ddof1"]), float(wa["ddof2"])
H, p_kw = stats.kruskal(*arrs)
w2 = omega_sq(arrs)

rec = dict(edge=EDGE, F=F, ddof1=ddof1, ddof2=ddof2, p=p_w,
           eta2=np2, omega2=w2, kw_H=H, kw_p=p_kw)
for g in GROUPS:
    gr = long[long.group == g]["r"]
    rec[f"mean_r_{g}"] = gr.mean(); rec[f"sd_r_{g}"] = gr.std(ddof=1); rec[f"n_{g}"] = len(gr)
omni = pd.DataFrame([rec])
omni.to_csv(f"{LANG_DIR}/group_language_putamen_omnibus.csv", index=False)

print(f"\nOmnibus Welch ANOVA on Fisher-z:")
print(f"  F({ddof1:.0f},{ddof2:.1f}) = {F:.3f},  p = {p_w:.4f} {stars(p_w)}"
      f"   eta^2={np2:.3f}  omega^2={w2:.3f}")
print(f"  Kruskal-Wallis H = {H:.3f}, p = {p_kw:.4f} {stars(p_kw)}")
print("  Group mean r (+/- SD):  " +
      "  ".join(f"{g}={rec[f'mean_r_{g}']:+.3f}±{rec[f'sd_r_{g}']:.3f}(n{rec[f'n_{g}']})"
               for g in GROUPS))

# ============================================================
# 3. Protected post-hoc (Games-Howell + Dunn)
# ============================================================
gh = pg.pairwise_gameshowell(data=long, dv="z", between="group")
gh_lu = {(a, b): row for (a, b), row in gh.set_index(["A", "B"]).iterrows()}
dunn = sp.posthoc_dunn(long, val_col="z", group_col="group", p_adjust="holm")

post = []
for A, B in CONTRASTS:
    row = gh_lu.get((A, B)) if (A, B) in gh_lu else gh_lu.get((B, A))
    sign = 1.0 if (A, B) in gh_lu else -1.0
    post.append(dict(contrast=f"{A}-{B}", diff_z=sign * float(row["diff"]),
                     hedges=sign * float(row["hedges"]), gh_T=sign * float(row["T"]),
                     gh_p=float(row["pval"]), dunn_p=float(dunn.loc[A, B])))
post = pd.DataFrame(post)
post["omnibus_p"] = p_w
post["protected"] = p_w < .05
post.to_csv(f"{LANG_DIR}/group_language_putamen_posthoc.csv", index=False)
print(f"\nSaved: {LANG_DIR}/group_language_putamen_omnibus.csv")
print(f"Saved: {LANG_DIR}/group_language_putamen_posthoc.csv")

if p_w < .05:
    print("\nProtected Games-Howell (omnibus significant):")
    print(post[["contrast", "diff_z", "hedges", "gh_p", "dunn_p"]].round(3).to_string(index=False))
else:
    print("\nOmnibus n.s. -> post-hoc not licensed (Games-Howell/Dunn computed but not interpreted):")
    print(post[["contrast", "hedges", "gh_p", "dunn_p"]].round(3).to_string(index=False))

# ============================================================
# 4. Figure: per-group distribution of the edge (r), box + jittered points + mean
# ============================================================
rng = np.random.default_rng(0)
fig, ax = plt.subplots(figsize=(6.4, 5.2))
data_by_g = [long[long.group == g]["r"].to_numpy() for g in GROUPS]
bp = ax.boxplot(data_by_g, positions=range(len(GROUPS)), widths=0.55,
                showfliers=False, patch_artist=True, zorder=1)
for patch, g in zip(bp["boxes"], GROUPS):
    patch.set_facecolor(GROUP_COLORS[g]); patch.set_alpha(0.25)
for med in bp["medians"]:
    med.set_color("black")
for k, g in enumerate(GROUPS):
    y = data_by_g[k]
    x = k + rng.uniform(-0.16, 0.16, len(y))
    ax.scatter(x, y, s=26, color=GROUP_COLORS[g], edgecolor="k", linewidth=0.3,
               alpha=0.85, zorder=3)
    ax.scatter([k], [y.mean()], marker="D", s=70, color="white",
               edgecolor=GROUP_COLORS[g], linewidth=2, zorder=4)
ax.axhline(0, color="grey", lw=0.8, ls="--")
ax.set_xticks(range(len(GROUPS)))
ax.set_xticklabels([f"{g}\n(n={counts.get(g,0)})" for g in GROUPS], fontweight="bold")
for tick, g in zip(ax.get_xticklabels(), GROUPS):
    tick.set_color(GROUP_COLORS[g])
ax.set_ylabel("Language frontal-putamen FC (Pearson's r)")
ax.set_title(f"Language frontal-putamen connectivity by group\n"
             f"{EDGE}\nWelch F({ddof1:.0f},{ddof2:.1f})={F:.2f}, p={p_w:.3f} {stars(p_w)}"
             f"  (ω²={w2:.3f})", fontsize=11)
ax.grid(axis="y", alpha=0.2)
plt.tight_layout()
plt.savefig(f"{LANG_DIR}/group_language_putamen.png", dpi=200, bbox_inches="tight")
plt.show()
print(f"\nSaved: {LANG_DIR}/group_language_putamen.png")


## 2. group x FC -> SDQ-emotional (negative binomial, with 95% CI)

NB model `emotional ~ C(group,Treatment('TD')) * FCz_c` on the single edge. Joint 2-df interaction LR + Freedman-Lane permutation; scatter now carries the delta-method 95% CI band on each group's predicted mean.

*Source: `stats/stats_language_connectivity_emotional_nb.py` (adapted for inline display).*

In [ ]:
"""
stats_language_connectivity_emotional_nb.py

## Author: Han Wang
### 2026-08-14: Initial version (language frontal-putamen arm).

Mood-coupling model for the SINGLE language frontal-putamen edge, matching the
striatal arm's stats_connectivity_emotional_nb.py: brain FC is the PREDICTOR and
SDQ-emotional the OUTCOME, one negative-binomial model, 3-level group (TD ref):

    emotional ~ C(group, Treatment('TD')) * FCz_c        [TD = reference]

Edge: Language-14_L-Ctx (L_44, pars opercularis) <-> Language-14_L-Putamen
      (medial/anterior left putamen).

  * Outcome = SDQ-emotional (0-10 count) -> negative binomial (log link).
  * Predictor = edge FC in Fisher-z (FCz), mean-centred, so group terms are the
    group-vs-TD emotional gaps at the mean connectivity.
  * Two interaction terms: [T.DLD]:FCz_c and [T.HSL]:FCz_c. Headline test is the
    JOINT likelihood-ratio test of BOTH (2 df: full vs additive). Also a
    distribution-free Freedman-Lane permutation p for that joint interaction.
    Single edge -> no multiple-comparison correction across tiles.

Input:  results/language_connectivity_outputs/group_language_putamen_long.csv
        dat_verbgen_analysis_144.csv  (emotional, group)
Output: results/language_connectivity_outputs/language_connectivity_emotional_nb_stats.csv
        results/language_connectivity_outputs/language_connectivity_emotional_nb_scatter.png
"""

import warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
import patsy

warnings.filterwarnings("ignore")
NPERM = 2000

PROJECT_DIR = "/home/hanwang/Apps/Programming/matlab-proj/PFM_MSHBM_MHVerbGen"
LANG_DIR = f"{PROJECT_DIR}/results/language_connectivity_outputs"
LISTCSV = ("/home/hanwang/Documents/Data/verb_gen_krishnan/"
           "behavioural_scq_sdq/dat_verbgen_analysis_144.csv")

GROUPS = ["DLD", "HSL", "TD"]
GROUP_COLORS = {"DLD": "#d63031", "HSL": "#2ca02c", "TD": "#0984e3"}
INTER = {"DLD": "C(group, Treatment('TD'))[T.DLD]:FCz_c",
         "HSL": "C(group, Treatment('TD'))[T.HSL]:FCz_c"}
MAIN = {"DLD": "C(group, Treatment('TD'))[T.DLD]",
        "HSL": "C(group, Treatment('TD'))[T.HSL]"}
CONTRASTS = ["DLD", "HSL"]
RNG = np.random.default_rng(0)
EDGE = "L-Putamen(Language) <-> L_44 (pars opercularis)"


def stars(p):
    return "***" if p < .001 else "**" if p < .01 else "*" if p < .05 else "n.s."


def pois_joint_lr(dd):
    f = smf.glm("emotional ~ C(group, Treatment('TD')) * FCz_c",
                data=dd, family=sm.families.Poisson()).fit()
    r = smf.glm("emotional ~ C(group, Treatment('TD')) + FCz_c",
                data=dd, family=sm.families.Poisson()).fit()
    return r.deviance - f.deviance


def freedman_lane_p(d):
    lr_obs = pois_joint_lr(d)
    red = smf.glm("emotional ~ C(group, Treatment('TD')) + FCz_c",
                  data=d, family=sm.families.Poisson()).fit()
    mu = red.fittedvalues.to_numpy()
    resid = d["emotional"].to_numpy() - mu
    dp = d.copy()
    n_ge = n_val = 0
    for _ in range(NPERM):
        dp["emotional"] = np.clip(np.round(mu + resid[RNG.permutation(len(resid))]), 0, None)
        try:
            n_ge += pois_joint_lr(dp) >= lr_obs
            n_val += 1
        except Exception:
            continue
    return (1 + n_ge) / (1 + n_val)


# ------------------------------------------------------------
# Data: long edge (z) + emotional outcome
# ------------------------------------------------------------
long = pd.read_csv(f"{LANG_DIR}/group_language_putamen_long.csv")
beh = pd.read_csv(LISTCSV)[["code", "emotional"]].copy()
beh["code"] = beh["code"].astype(str)
long["code"] = long["code"].astype(str)
d = long.merge(beh, on="code", how="left").dropna(subset=["emotional"])
d["emotional"] = d["emotional"].round().astype(int).clip(0, 10)
d = d.rename(columns={"z": "FCz"})
d["FCz_c"] = d["FCz"] - d["FCz"].mean()
counts = d["group"].value_counts().to_dict()
print(f"{len(d)} subjects | " + ", ".join(f"{g}={counts.get(g,0)}" for g in GROUPS))

# ------------------------------------------------------------
# One NB model, 3-level group
# ------------------------------------------------------------
full = smf.negativebinomial(
    "emotional ~ C(group, Treatment('TD')) * FCz_c", data=d).fit(disp=0)
red = smf.negativebinomial(
    "emotional ~ C(group, Treatment('TD')) + FCz_c", data=d).fit(disp=0)
lr = 2 * (full.llf - red.llf)
lr_p = stats.chi2.sf(lr, 2)
perm_p = freedman_lane_p(d)

rec = dict(edge=EDGE,
           slope_TD=full.params["FCz_c"],
           slope_DLD=full.params["FCz_c"] + full.params[INTER["DLD"]],
           slope_HSL=full.params["FCz_c"] + full.params[INTER["HSL"]],
           inter_beta_DLD=full.params[INTER["DLD"]], inter_z_DLD=full.tvalues[INTER["DLD"]],
           inter_wald_p_DLD=full.pvalues[INTER["DLD"]],
           inter_beta_HSL=full.params[INTER["HSL"]], inter_z_HSL=full.tvalues[INTER["HSL"]],
           inter_wald_p_HSL=full.pvalues[INTER["HSL"]],
           inter_lr_chi2=lr, inter_lr_p=lr_p, inter_perm_p=perm_p,
           alpha=full.params["alpha"])
stat = pd.DataFrame([rec])
stat.to_csv(f"{LANG_DIR}/language_connectivity_emotional_nb_stats.csv", index=False)
print(f"Saved: {LANG_DIR}/language_connectivity_emotional_nb_stats.csv\n")

print(f"NB  emotional ~ group * FCz   ({EDGE})")
print(f"  FC->emotional slope (log-count):  TD={rec['slope_TD']:+.3f}  "
      f"DLD={rec['slope_DLD']:+.3f}  HSL={rec['slope_HSL']:+.3f}")
print(f"  interaction [DLD-TD]: beta={rec['inter_beta_DLD']:+.3f}  z={rec['inter_z_DLD']:+.2f}  "
      f"Wald p={rec['inter_wald_p_DLD']:.3f} {stars(rec['inter_wald_p_DLD'])}")
print(f"  interaction [HSL-TD]: beta={rec['inter_beta_HSL']:+.3f}  z={rec['inter_z_HSL']:+.2f}  "
      f"Wald p={rec['inter_wald_p_HSL']:.3f} {stars(rec['inter_wald_p_HSL'])}")
print(f"  JOINT interaction LR chi2(2)={lr:.3f}, p={lr_p:.4f} {stars(lr_p)}  |  "
      f"Freedman-Lane perm p={perm_p:.4f} {stars(perm_p)}")

# ------------------------------------------------------------
# Predicted mean + 95% CI per group (delta method on linear predictor):
# var(eta) = x' Cov(beta) x on the log-count scale, back-transformed by exp().
# Mirrors the network-size NB scripts (stats_emotional_*_nb_interaction.py).
# ------------------------------------------------------------
zmean = d["FCz"].mean()
design_info = full.model.data.design_info
kf = len(design_info.column_names)
beta = np.asarray(full.params)[:kf]          # mean-structure betas (alpha is last)
cov = np.asarray(full.cov_params())[:kf, :kf]


def predict_band(group, n=120):
    dg = d[d.group == group]
    xs = np.linspace(dg["FCz"].min(), dg["FCz"].max(), n)
    grid = pd.DataFrame({"group": group, "FCz": xs, "FCz_c": xs - zmean})
    X = np.asarray(patsy.dmatrix(design_info, grid))
    eta = X @ beta
    se = np.sqrt(np.einsum("ij,jk,ik->i", X, cov, X))
    return xs, np.exp(eta), np.exp(eta - 1.96 * se), np.exp(eta + 1.96 * se)


# ------------------------------------------------------------
# Figure: scatter (x=FC z, y=emotional) + NB predicted mean +/- 95% CI, 3 groups
# ------------------------------------------------------------
fig, ax = plt.subplots(figsize=(7.2, 5.6))
pred_rows = []
for g in GROUPS:
    dg = d[d.group == g]
    c = GROUP_COLORS[g]
    xs, mu, lo, hi = predict_band(g)
    ax.fill_between(xs, lo, hi, color=c, alpha=0.15, lw=0)
    ax.plot(xs, mu, color=c, lw=2.2, label=f"{g} (n={counts.get(g,0)})")
    yj = dg["emotional"] + RNG.uniform(-0.15, 0.15, len(dg))
    ax.scatter(dg["FCz"], yj, s=30, alpha=0.8, color=c,
               edgecolor="k", linewidth=0.3)
    for xv, mv, lv, hv in zip(xs, mu, lo, hi):
        pred_rows.append(dict(group=g, FCz=round(xv, 4), mean=round(mv, 4),
                              ci_lo=round(lv, 4), ci_hi=round(hv, 4)))
ax.set_ylim(-0.5, 10.5)
ax.set_xlabel("Language frontal-putamen FC (Fisher z)")
ax.set_ylabel("SDQ emotional")
ax.legend(fontsize=9, loc="best")
ax.grid(alpha=0.2)
ax.set_title("NB-predicted SDQ-emotional vs language frontal-putamen FC, by group\n"
             "shaded = 95% CI on predicted mean;  "
             f"joint interaction LR p={lr_p:.3f} {stars(lr_p)}  "
             f"(perm p={perm_p:.3f})", fontsize=11)
plt.tight_layout()
plt.savefig(f"{LANG_DIR}/language_connectivity_emotional_nb_scatter.png",
            dpi=200, bbox_inches="tight")
plt.show()
print(f"\nSaved: {LANG_DIR}/language_connectivity_emotional_nb_scatter.png")

pd.DataFrame(pred_rows).to_csv(
    f"{LANG_DIR}/language_connectivity_emotional_nb_predband.csv", index=False)
print(f"Saved: {LANG_DIR}/language_connectivity_emotional_nb_predband.csv")


## 3. group x Language network size -> SDQ-emotional (negative binomial)

NB model `emotional ~ C(group,Treatment('TD')) * language_c` with the predicted mean +/- 95% CI band already built in.

*Source: `stats/stats_emotional_language_nb_interaction.py` (adapted for inline display).*

In [ ]:
"""
stats_emotional_language_nb_interaction.py

Negative-binomial model of SDQ-emotional symptoms on Language-network size with a
group x language interaction, now with the 3-level group factor (TD reference)
over all 144 subjects:

    emotional ~ C(group, Treatment('TD')) * language_c      (log link, NB2)

The log link keeps the mean non-negative (handles the floor at 0 without a curve)
and the NB dispersion soaks up the DLD over-dispersion. TD's language slope is the
Lynch-style positive control (higher language -> more emotional symptoms in
controls); the two interaction terms test whether DLD and HSL depart from it.

Robustness of the interaction:
  (A) NB joint likelihood-ratio test of BOTH interaction terms (2 df).
  (B) Freedman-Lane permutation using a Poisson working model, statistic = the
      joint interaction LR (deviance drop, full vs additive) -- distribution-free.

Variant selects which MS-HBM set the Language size comes from (full / icafix).

Input:  results/network_size_<variant>/group_network_size_long.csv (Language size)
        dat_verbgen_analysis_144.csv                                (emotional)
Output: results/network_size_<variant>/emotional_language_nb_interaction.csv
        results/network_size_<variant>/emotional_language_interaction_robustness.csv
        results/network_size_<variant>/emotional_vs_language_nb.png

## Author: Han Wang
"""

import argparse
import warnings
import numpy as np
import pandas as pd
import patsy
import matplotlib
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats

warnings.filterwarnings("ignore")

PROJECT_DIR = "/home/hanwang/Apps/Programming/matlab-proj/PFM_MSHBM_MHVerbGen"
LISTCSV = ("/home/hanwang/Documents/Data/verb_gen_krishnan/"
           "behavioural_scq_sdq/dat_verbgen_analysis_144.csv")
GROUPS = ["DLD", "HSL", "TD"]
GROUP_COLORS = {"DLD": "#d63031", "HSL": "#2ca02c", "TD": "#0984e3"}
INTER = {"DLD": "C(group, Treatment('TD'))[T.DLD]:language_c",
         "HSL": "C(group, Treatment('TD'))[T.HSL]:language_c"}
MAIN = {"DLD": "C(group, Treatment('TD'))[T.DLD]",
        "HSL": "C(group, Treatment('TD'))[T.HSL]"}
RNG = np.random.default_rng(0)

class _Args:  # notebook shim (was argparse) -- change variant here
    variant = "full"
args = _Args()
NS_DIR = f"{PROJECT_DIR}/results/network_size_{args.variant}"

# ------------------------------------------------------------
# Data
# ------------------------------------------------------------
beh = pd.read_csv(LISTCSV)[["code", "emotional"]]
beh["code"] = beh["code"].astype(str)
long = pd.read_csv(f"{NS_DIR}/group_network_size_long.csv")
long["code"] = long["code"].astype(str)
sal = long[long["network_label"] == "Language"][["code", "group", "network_size_pct"]]
dat = (sal.merge(beh, on="code", how="left")
          .rename(columns={"network_size_pct": "language"})
          .dropna(subset=["emotional", "language"]))
dat = dat[dat["group"].isin(GROUPS)].copy()
dat["emotional"] = dat["emotional"].round().astype(int).clip(0, 10)
sal_mean = dat["language"].mean()
dat["language_c"] = dat["language"] - sal_mean
print(f"[{args.variant}] n={len(dat)} | "
      + dat["group"].value_counts().to_dict().__str__())

# ------------------------------------------------------------
# (A) NB interaction model, 3-level group
# ------------------------------------------------------------
F_FULL = "emotional ~ C(group, Treatment('TD')) * language_c"
F_RED = "emotional ~ C(group, Treatment('TD')) + language_c"
nb_full = smf.negativebinomial(F_FULL, data=dat).fit(disp=0)
nb_red = smf.negativebinomial(F_RED, data=dat).fit(disp=0)
lr_stat = 2 * (nb_full.llf - nb_red.llf)
lr_p = stats.chi2.sf(lr_stat, 2)              # joint: both interaction terms

sl = {"TD": nb_full.params["language_c"]}
sl["DLD"] = sl["TD"] + nb_full.params[INTER["DLD"]]
sl["HSL"] = sl["TD"] + nb_full.params[INTER["HSL"]]

print("=" * 70)
print(f"NB: emotional ~ group * language_c  [TD ref]  (variant={args.variant})")
print("=" * 70)
print("Language slope (log-count) per group:")
for g in GROUPS:
    print(f"  {g}: {sl[g]:+.4f}  (RR={np.exp(sl[g]):.3f} per +1% cortex)")
print(f"TD slope (positive control) Wald p = {nb_full.pvalues['language_c']:.4g}")
for g in ["DLD", "HSL"]:
    print(f"  interaction {g}-TD = {nb_full.params[INTER[g]]:+.4f}, "
          f"Wald p = {nb_full.pvalues[INTER[g]]:.4g}")
print(f"Joint interaction LR chi2(2) = {lr_stat:.3f}, p = {lr_p:.4g}")

# ------------------------------------------------------------
# (B) Freedman-Lane permutation (Poisson working model; joint LR statistic)
# ------------------------------------------------------------
def pois_joint_lr(data):
    f = smf.glm(F_FULL, data=data, family=sm.families.Poisson()).fit()
    r = smf.glm(F_RED, data=data, family=sm.families.Poisson()).fit()
    return r.deviance - f.deviance          # = 2*(llf_full - llf_red)

lr_obs = pois_joint_lr(dat)
pois_red = smf.glm(F_RED, data=dat, family=sm.families.Poisson()).fit()
mu_red = pois_red.fittedvalues.to_numpy()
resid = dat["emotional"].to_numpy() - mu_red
NPERM = 2000
dperm = dat.copy()
n_ge = n_valid = 0
for _ in range(NPERM):
    dperm["emotional"] = np.clip(np.round(mu_red + resid[RNG.permutation(len(resid))]), 0, None)
    try:
        lrp = pois_joint_lr(dperm)
    except Exception:
        continue
    n_valid += 1
    n_ge += lrp >= lr_obs
p_perm = (1 + n_ge) / (1 + n_valid)

print("\n" + "-" * 70)
print("ROBUSTNESS — joint group x language interaction")
print(f"  (A) NB likelihood-ratio        chi2(2) = {lr_stat:5.3f}   p = {lr_p:.4f}")
print(f"  (B) Freedman-Lane permutation  ({n_valid} perms)      p = {p_perm:.4f}")

pd.DataFrame([
    dict(test="NB_joint_LR_chi2_2df", statistic=round(lr_stat, 3), p=round(lr_p, 4)),
    dict(test="Freedman_Lane_permutation", statistic=n_valid, p=round(p_perm, 4)),
]).to_csv(f"{NS_DIR}/emotional_language_interaction_robustness.csv", index=False)
print(f"Saved: {NS_DIR}/emotional_language_interaction_robustness.csv")

# ------------------------------------------------------------
# Predicted mean + 95% CI per group (delta method on linear predictor)
# ------------------------------------------------------------
design_info = nb_full.model.data.design_info
k = len(design_info.column_names)
beta = np.asarray(nb_full.params)[:k]
cov = np.asarray(nb_full.cov_params())[:k, :k]


def predict_band(group, n=200):
    d = dat[dat["group"] == group]
    xs = np.linspace(d["language"].min(), d["language"].max(), n)
    grid = pd.DataFrame({"group": group, "language": xs, "language_c": xs - sal_mean})
    X = np.asarray(patsy.dmatrix(design_info, grid))
    eta = X @ beta
    se = np.sqrt(np.einsum("ij,jk,ik->i", X, cov, X))
    return xs, np.exp(eta), np.exp(eta - 1.96 * se), np.exp(eta + 1.96 * se)


pred_rows = []
fig, ax = plt.subplots(figsize=(7.4, 5.6))
for g in GROUPS:
    d = dat[dat["group"] == g]
    c = GROUP_COLORS[g]
    xs, mu, lo, hi = predict_band(g)
    ax.fill_between(xs, lo, hi, color=c, alpha=0.15, lw=0)
    ax.plot(xs, mu, color=c, lw=2.4,
            label=f"{g} (n={len(d)}): RR={np.exp(sl[g]):.2f}/+1%")
    yj = d["emotional"] + RNG.uniform(-0.12, 0.12, len(d))
    ax.scatter(d["language"], yj, color=c, edgecolor="k", linewidth=0.3,
               s=34, zorder=3, alpha=0.85)
    for xv, mv, lv, hv in zip(xs, mu, lo, hi):
        pred_rows.append(dict(group=g, language=round(xv, 4), mean=round(mv, 4),
                              ci_lo=round(lv, 4), ci_hi=round(hv, 4)))

ax.set_xlabel("Language network size (% cortical surface)")
ax.set_ylabel("SDQ emotional symptoms (0-10)")
ax.set_ylim(-0.5, 10.5)
ax.set_title("Emotional symptoms vs Language size — NB fit (mean ± 95% CI)\n"
             f"joint group × language interaction LR p = {lr_p:.3f} "
             f"(variant={args.variant})", fontsize=12)
ax.grid(alpha=0.25); ax.set_axisbelow(True)
ax.legend(title="NB predicted mean", fontsize=9, loc="upper center")
plt.tight_layout()
out_png = f"{NS_DIR}/emotional_vs_language_nb.png"
plt.savefig(out_png, dpi=200, bbox_inches="tight")
plt.show()

pd.DataFrame(pred_rows).to_csv(
    f"{NS_DIR}/emotional_language_nb_interaction.csv", index=False)
print(f"\nSaved: {out_png}")
print(f"Saved: {NS_DIR}/emotional_language_nb_interaction.csv")
